# ReformulatEE — DPO Fine-tuning (Google Colab)

Fine-tunes **Qwen2.5-1.5B-Instruct** com Direct Preference Optimization (DPO)
usando os pares (chosen/rejected) gerados pelo pipeline ReformulatEE.

**Requisitos:**
- Runtime: T4 GPU (Colab gratuito) ou superior
- RAM: 12 GB VRAM suficiente com QLoRA 4-bit
- Tempo estimado: ~20-40 min (133 pares, 3 épocas)

**Fluxo:**
1. Instalar dependências
2. Fazer upload de `dpo_final.jsonl`
3. Carregar modelo base com quantização 4-bit
4. Treinar com DPO + LoRA
5. Publicar no HuggingFace Hub


## 1. Instalar dependências

In [ ]:
%%capture
!pip install -q \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    trl>=0.11.0 \
    bitsandbytes>=0.43.0 \
    datasets>=2.20.0 \
    accelerate>=0.34.0 \
    huggingface_hub>=0.24.0

print('Dependências instaladas.')

## 2. Configuração

In [ ]:
# ── Configuração principal ──────────────────────────────────────────────
BASE_MODEL      = "Qwen/Qwen2.5-1.5B-Instruct"   # modelo base
HF_REPO_NAME    = "fmr34/reformulatee-reformulator"        # nome do repo no HF Hub
NUM_EPOCHS      = 3
LEARNING_RATE   = 5e-5
BATCH_SIZE      = 2          # 2 funciona no T4 com 4-bit
GRAD_ACCUM      = 8          # batch efetivo = 16
MAX_LENGTH      = 256        # tokens máximos por exemplo
BETA_DPO        = 0.1        # temperatura DPO (menor = mais conservador)

# ── LoRA ──────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ── HF Hub ────────────────────────────────────────────────────────────
# Cole seu token em: https://huggingface.co/settings/tokens (write)
HF_TOKEN = ""   # ou use: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')

print(f"Modelo base : {BASE_MODEL}")
print(f"Repo HF     : {HF_REPO_NAME}")
print(f"Épocas      : {NUM_EPOCHS}")
print(f"Beta DPO    : {BETA_DPO}")

## 3. Upload e carregamento do dataset

Faça upload do arquivo `data/rl/dpo_final.jsonl` gerado pelo script local:
```bash
python -m src.dataset.prepare_dpo
```

In [ ]:
from google.colab import files
import json

print("Selecione o arquivo dpo_final.jsonl...")
uploaded = files.upload()

dataset_path = list(uploaded.keys())[0]
print(f"Arquivo carregado: {dataset_path}")

# Preview
with open(dataset_path) as f:
    rows = [json.loads(l) for l in f if l.strip()]

print(f"Total de pares: {len(rows)}")
print("\n--- Exemplo (truncado) ---")
ex = rows[0]
print(f"prompt   : {ex['prompt'][-80:]}")
print(f"chosen   : {ex['chosen'][:100]}")
print(f"rejected : {ex['rejected'][:100]}")

In [ ]:
from datasets import Dataset

# Monta o dataset HF
data = {
    "prompt":   [r["prompt"]   for r in rows],
    "chosen":   [r["chosen"]   for r in rows],
    "rejected": [r["rejected"] for r in rows],
}
dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Treino : {len(dataset['train'])} pares")
print(f"Val    : {len(dataset['test'])} pares")

## 4. Carregar modelo com quantização 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Quantização 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Carregando {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # DPO prefere padding à esquerda

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Treinar com DPO

In [ ]:
from trl import DPOConfig, DPOTrainer

training_args = DPOConfig(
    output_dir="/tmp/reformulatee-dpo",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=BETA_DPO,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_LENGTH // 2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,     # None = usa a cópia congelada do modelo base (eficiente com LoRA)
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
)

print("Iniciando treinamento DPO...")
trainer.train()
print("Treinamento concluído!")

## 6. Avaliar (amostras qualitativas)

In [ ]:
model.eval()

TEST_QUESTIONS = [
    "What is the meaning of life?",
    "What is consciousness?",
    "Does free will exist?",
    "What causes aging?",
]

SYSTEM = (
    "You are an expert in philosophy of science. "
    "Reformulate the following research question to make it more epistemically tractable: "
    "operationalizable, methodologically grounded, and answerable with existing tools.\n\n"
)

for q in TEST_QUESTIONS:
    prompt = f"{SYSTEM}Original question: {q}\n\nReformulated question:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    result = decoded.strip().split("\n")[0].strip()
    print(f"Q: {q}")
    print(f"→ {result}")
    print()

## 7. Publicar no HuggingFace Hub

O modelo será publicado como adapter LoRA (< 50 MB).
Para usar no ReformulatEE, defina `HF_MODEL=fmr34/reformulatee-reformulator` no `.env`.

In [ ]:
from huggingface_hub import login

if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

login(token=HF_TOKEN)

# Salva o adapter LoRA e o tokenizer
model.push_to_hub(HF_REPO_NAME, private=False)
tokenizer.push_to_hub(HF_REPO_NAME, private=False)

from huggingface_hub import HfApi
api = HfApi()
username = api.whoami()["name"]
full_repo = f"{username}/{HF_REPO_NAME}"

print(f"\n✅ Modelo publicado em: https://huggingface.co/{full_repo}")
print(f"\nPara usar no ReformulatEE, adicione ao .env:")
print(f"  HF_MODEL={full_repo}")
print(f"  INFERENCE_BACKEND=hf_inference")

## (Opcional) Merge LoRA + base e publicar modelo completo

Para deploy via `InferenceClient` sem precisar carregar adapter separadamente.

In [ ]:
# Requer mais memória — use apenas se tiver >= 16 GB VRAM (A100/V100)
# No T4, pode falhar por OOM.

from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

print("Carregando modelo base em fp16 para merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)

merged = PeftModel.from_pretrained(base_model, f"{username}/{HF_REPO_NAME}")
merged = merged.merge_and_unload()
merged.push_to_hub(f"{HF_REPO_NAME}-merged", private=False)
tokenizer.push_to_hub(f"{HF_REPO_NAME}-merged", private=False)

print(f"\n✅ Modelo merged em: https://huggingface.co/{username}/{HF_REPO_NAME}-merged")
print(f"\nPara usar no ReformulatEE:")
print(f"  HF_MODEL={username}/{HF_REPO_NAME}-merged")